# Chapter 7 Exercise — Coffee Maker Execution

Apply sympy symbolic binding, state machine traces, and a parameter sweep to the coffee maker model from Chapter 6.

## Problem

Your model from Chapter 6 has a `BrewUnit` with a `throughput : Real default = 0.3` attribute. Work through these steps:

1. Define a sympy expression for brew energy: `Q = power * duration * efficiency`,    lambdify it, and verify the reference value for a 1200 W element running for 90 s    at 0.65 efficiency. Expected: `1200 × 90 × 0.65 = 70200 J`. Assert within 1 J.

2. Add a `state BrewCycle` to your model with four substates (`idle`, `brewing`,    `done`, `fault`) and three transitions:
   - `idle → brewing` on `Start`
   - `brewing → done` on `Finish`
   - `brewing → fault` on `Overheat`
   Verify with `execute_state` that `events=['Start', 'Finish']` visits `['idle', 'brewing', 'done']`.

3. Use `sweep_1d` to sweep brew duration from 60 s to 180 s at fixed power (1200 W)    and efficiency (0.65). Plot energy vs. duration and mark a 40 kJ threshold.    Print the minimum duration that meets the threshold.

Verify: reference value matches within 1 J; normal trace is `['idle', 'brewing', 'done']`; threshold duration is printed.


In [ ]:
import opensysml
import sympy as sp
import numpy as np
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your Chapter 6 coffee maker model here, then add BrewCycle.
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")
if not model.ok:
    print(format_diagnostics(model.diagnostics))


In [ ]:
# Step 1: sympy binding and reference value check
P, t, eta = sp.symbols('P t eta', positive=True)
Q_fn = sp.lambdify([P, t, eta], P * t * eta, 'numpy')

ref = float(Q_fn(1200.0, 90.0, 0.65))
assert abs(ref - 70200.0) < 1.0, f"Reference mismatch: {ref}"
print(f"Q_fn(1200, 90, 0.65) = {ref:.1f} J  (expected 70200.0)")


In [ ]:
# Step 2: state machine trace
brew_cycle = model.find("CoffeeDemo::BrewCycle")
if brew_cycle:
    normal = model.execute_state(brew_cycle.id, events=["Start", "Finish"])
    print(f"Normal trace: {normal['states_visited']}")
    assert normal["states_visited"] == ["idle", "brewing", "done"]
else:
    print("BrewCycle not found — check the qualified name")


In [ ]:
# Step 3: parameter sweep over brew duration
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from toaster.simulate import sweep_1d

t_vals = np.linspace(60, 180, 60)
Q_vals = sweep_1d(Q_fn, t_vals, P=1200.0, eta=0.65)

threshold = 40_000.0
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t_vals, Q_vals / 1000, label="Brew energy")
ax.axhline(threshold / 1000, color="red", linestyle="--",
           label=f"Threshold {threshold/1000:.0f} kJ")
ax.set_xlabel("Brew duration (s)")
ax.set_ylabel("Energy (kJ)")
ax.set_title("Brew energy vs. duration (P=1200 W, η=0.65)")
ax.legend()
fig.tight_layout()
fig.savefig("ch07_exercise_sweep.svg")
plt.close(fig)

crossing_idx = np.argmax(Q_vals >= threshold)
print(f"Minimum duration to meet threshold: {t_vals[crossing_idx]:.1f} s")
conn.close()
